In [1]:
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Ẩn các thông báo Log của TensorFlow
import tensorflow as tf

E0000 00:00:1774270365.803743      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774270365.864071      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774270366.308898      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774270366.308935      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774270366.308937      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774270366.308940      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

# 1. Định nghĩa tham số cố định
TRAIN_PATH = '/kaggle/input/datasets/jaquy0802/edge-ai/kaggle_testing/train'
IMG_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10

print("Đã sẵn sàng thiết lập!")

Đã sẵn sàng thiết lập!


In [3]:
# Nạp tập Train (80% để huấn luyện)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb'
)

# Nạp tập Validation (20% để kiểm tra độ chính xác)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb'
)

class_names = train_ds.class_names
print(f"Danh sách nhãn: {class_names}")

# Tối ưu tốc độ nạp dữ liệu vào RAM
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 2000 files belonging to 10 classes.
Using 1600 files for training.


I0000 00:00:1774270394.131791      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774270394.137769      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 2000 files belonging to 10 classes.
Using 400 files for validation.
Danh sách nhãn: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [4]:
from tensorflow.keras import layers, models, callbacks

# 1. Giữ nguyên Augmentation để mô hình bền bỉ
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.15),
  layers.RandomZoom(0.15),
])

# 2. Xây dựng kiến trúc mô hình (~189,450 tham số)
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    
    # Block 1
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(), # Giúp học nhanh và ổn định hơn
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Block 2
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Block 3
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    # Tăng cường lớp Dense để tận dụng nốt 77k tham số còn lại
    layers.Dense(128, activation='relu'), 
    layers.Dropout(0.3), # Chống học vẹt (overfitting)
    layers.Dense(10, activation='softmax')
])

model.summary() # Huy hãy check dòng Total Params để thấy độ sát sao


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,450 (740.04 KB)

 Trainable params: 189,130 (738.79 KB)

 Non-trainable params: 320 (1.25 KB)

In [5]:
# Tự động giảm tốc độ học khi độ chính xác chững lại
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Dừng sớm nếu không còn tiến bộ và lấy lại bản tốt nhất
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Mạnh dạn nâng Epochs lên 50, Early Stopping sẽ lo việc dừng đúng lúc
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[reduce_lr, early_stop]
)

Epoch 1/50


I0000 00:00:1774270402.868846      76 cuda_dnn.cc:529] Loaded cuDNN version 91002


50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.2017 - loss: 2.4048 - val_accuracy: 0.1150 - val_loss: 2.2177 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4734 - loss: 1.4503 - val_accuracy: 0.1800 - val_loss: 2.2314 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5819 - loss: 1.1092 - val_accuracy: 0.2025 - val_loss: 2.3230 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6368 - loss: 0.9403 - val_accuracy: 0.2950 - val_loss: 2.0104 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7318 - loss: 0.7280 - val_accuracy: 0.3975 - val_loss: 1.6713 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7599 - loss: 0.6676 - val_accuracy: 0.5375 - val_loss: 1.2636 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7961 - loss: 0.5688 - val_accuracy: 0.6750 - 

In [6]:
# 1. Lưu file gốc .h5 (Để nộp cho BTC kiểm tra tham số)
model.save('edge_ai_model_final.h5')

# 2. Chuyển đổi sang .tflite (Để nạp vào ESP32-S3)
# Sử dụng Integer Quantization để nén mô hình tối đa
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Lưu file .tflite vào thư mục làm việc của Kaggle
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

print("Đã lưu xong 2 file: edge_ai_model_final.h5 và model_quantized.tflite!")

INFO:tensorflow:Assets written to: /tmp/tmpe9a8pcgo/assets


INFO:tensorflow:Assets written to: /tmp/tmpe9a8pcgo/assets


Saved artifact at '/tmp/tmpe9a8pcgo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135254768571920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765258896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765258320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765260048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765258128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765260624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765261392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765261968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765262352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765263312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135254765263120

W0000 00:00:1774270420.775168      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774270420.775201      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


Đã lưu xong 2 file: edge_ai_model_final.h5 và model_quantized.tflite!


I0000 00:00:1774270420.789923      23 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


In [7]:
import numpy as np
import os
import cv2
import pandas as pd
import tensorflow as tf

# 1. Định nghĩa đường dẫn và cấu hình (Sửa lại cho đúng Dataset của bạn)
TEST_DIR = '/kaggle/input/datasets/jaquy0802/edge-ai/kaggle_testing/test'
IMG_SIZE = (32, 32) # Phải trùng với kích thước lúc train

print("Đã thiết lập đường dẫn tập Test!")

Đã thiết lập đường dẫn tập Test!


In [8]:
import cv2
import numpy as np
import os

# 1. Sắp xếp file theo số (0.png, 1.png...) để khớp với ID nộp bài
test_files = sorted(os.listdir(TEST_DIR), key=lambda x: int(x.split('.')[0]))

X_test = []

print("Bắt đầu xử lý lại ảnh Test (Đã bỏ chia 255)...")

for f in test_files:
    img_path = os.path.join(TEST_DIR, f)
    img = cv2.imread(img_path)
    if img is not None:
        # Chuyển từ BGR sang RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Resize về đúng kích thước 32x32
        img = cv2.resize(img, IMG_SIZE)
        
        # QUAN TRỌNG: Giữ nguyên giá trị pixel 0-255. 
        # Không chia cho 255 ở đây vì mô hình đã có lớp Rescaling(1./255) rồi.
        X_test.append(img)

# Chuyển thành mảng Numpy
X_test = np.array(X_test)

print(f"Thành công: Đã xử lý {len(X_test)} ảnh Test.")
print(f"Kích thước mảng đầu vào: {X_test.shape}") # Kết quả nên là (1000, 32, 32, 3)

Bắt đầu xử lý lại ảnh Test (Đã bỏ chia 255)...
Thành công: Đã xử lý 1000 ảnh Test.
Kích thước mảng đầu vào: (1000, 32, 32, 3)


In [9]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os

print("Đang dự đoán bằng .tflite và khớp ID theo tên file thực tế...")

# 1. Nạp mô hình TFLite
interpreter = tf.lite.Interpreter(model_path="model_quantized.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 2. Chuẩn bị danh sách ID thực tế từ tên file đã sắp xếp
# Giả sử test_files đã được sorted ở ô trước đó
actual_ids = []
for f in test_files:
    file_name = f.split('.')[0] # Lấy phần số: "7" từ "7.png"
    actual_ids.append(f"{int(file_name):05d}") # Chuyển thành "00007"

final_labels_tflite = []

# 3. Chạy dự đoán từng ảnh
for i in range(len(X_test)):
    input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    output_data = interpreter.get_tensor(output_details[0]['index'])
    label = np.argmax(output_data)
    final_labels_tflite.append(label)

# 4. Tạo file submission với ID chuẩn từ tên file
submission = pd.DataFrame({
    'Id': actual_ids, # Dùng danh sách ID thực tế thay vì dùng range()
    'Label': final_labels_tflite
})

submission.to_csv('submission.csv', index=False)

print("--- THÀNH CÔNG ---")
print(f"Đã tạo xong file submission.csv với {len(submission)} dòng.")
print("Ví dụ 5 dòng đầu tiên:")
print(submission.head())

Đang dự đoán bằng .tflite và khớp ID theo tên file thực tế...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


--- THÀNH CÔNG ---
Đã tạo xong file submission.csv với 1000 dòng.
Ví dụ 5 dòng đầu tiên:
      Id  Label
0  00000      5
1  00007      2
2  00012      2
3  00014      5
4  00022      1
